In [1]:
import urllib.request  
import re 
import random
import numpy as np
import pandas as pd
from corus import load_lenta  
from tqdm import tqdm  
from pymorphy3 import MorphAnalyzer 

from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from sentence_transformers import SentenceTransformer
from umap import UMAP
import hdbscan
from sklearn.feature_extraction.text import CountVectorizer

from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
import nltk
from nltk.corpus import stopwords


SEED = 12
random.seed(SEED)
np.random.seed(SEED)


In [2]:
# Скачиваем архив с данными, если его нет в локальной папке
data_url = "https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz"
local_file = "lenta-ru-news.csv.gz"
try:
    open(local_file, 'rb').close()
except FileNotFoundError:
    print("Скачиваем набор данных...")
    urllib.request.urlretrieve(data_url, local_file)
    print("Скачивание завершено.")

In [3]:
raw = load_lenta(local_file)
news_df = pd.DataFrame(raw, columns=['url','title','text','topic','tags','date']) # Оставляем только заголовки и тексты, берём случайную подвыборку
news_df = news_df[['title','text','topic']].sample(n=100_000, random_state=SEED).reset_index(drop=True)
documents = (news_df['title'] + ' ' + news_df['text']).tolist() # Соединяем поля для обработки

Предобработка текста - лемматизация и удаление шума:
- Приведение к нижнему регистру снижает размер словаря за счёт унификации форм
- Удаление HTML-тэгов и неалфавитных символов устраняет шум
- Исключение стоп-слов убирает часто встречающиеся нерелевантные слова
- Лемматизация через pymorphy3 - слова к норме, улучшая сопоставимость токенов

In [4]:
# Загрузка стоп-слов и инициализация MorphAnalyzer()
nltk.download('stopwords', quiet=True)
russian_stop = set(stopwords.words('russian'))
morph = MorphAnalyzer()

# Функция для очистки и лемматизации
def clean_text(doc: str) -> str:
    # Приводим к единому регистру
    doc = doc.lower()
    # Убираем html-тэги
    doc = re.sub(r'<[^>]+>', ' ', doc)
    # Оставляем буквы и пробелы
    doc = re.sub(r'[^а-яa-zё\s]', ' ', doc)
    # Быстрая токенизация и удаление стоп-слов
    tokens = [w for w in doc.split() if w not in russian_stop]
    # Лемматизация через pymorphy3
    lemmas = [morph.parse(w)[0].normal_form for w in tokens]
    return ' '.join(lemmas)

In [5]:
# Применяем препроцессинг ко всем документам
cleaned_docs = []
for text in tqdm(documents, desc="Предобработка"):  # удаляем шум, приводим к леммам
    cleaned_docs.append(clean_text(text))

Предобработка: 100%|██████████| 100000/100000 [13:54<00:00, 119.81it/s]


## Построение пайплайна BERTopic

In [6]:
# Эмбеддинги (русскоязычная модель)
embedder = SentenceTransformer("DeepPavlov/rubert-base-cased")  # модель для русского языка

No sentence-transformers model found with name DeepPavlov/rubert-base-cased. Creating a new one with mean pooling.
Some weights of the model checkpoint at DeepPavlov/rubert-base-cased were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model

Снижение размерности перед кластеризацией

UMAP — сохраняет как локальную, так и глобальную структуру данных при понижении размерности, что важно для стабильной работы HDBSCAN
- n_neighbors=20 - оптимальное сочетание сохранения локальных соседей и глобальной структуры
- n_components=5 - достаточная размерность для разделения топиков перед визуализацией
- min_dist=0.05 - позволяет точкам образовывать плотные кластеры без сильного размывания

In [7]:
umap_model = UMAP(
    n_neighbors=20,       # ширина локального «окна»
    n_components=5,       # размерность для кластеризации
    min_dist=0.05,        # плотность представления в пространстве
    metric='cosine',
    random_state=SEED
)

Кластеризация

HDBSCAN — плотностной кластеризатор, автоматически определяющий количество тем и фильтрующий шум
- min_cluster_size=60 - минимальный объём кластера, позволяющий отбросить редкие шумовые темы
- metric='manhattan'- устойчив к выбросам в эмбеддинговом пространстве

In [8]:
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=60,  # минимальное число точек в кластере
    metric='manhattan',   # мера расстояния между embeddings
    prediction_data=True
)

Векторизация текстов (униграммы + биграммы)

CountVectorizer(ngram_range=(1,2), min_df=10, max_df=0.85) — извлечение униграмм и биграмм, отсекает очень редкие (<10 документов) и слишком частые (>85% документов) токены для повышения информативности тем

In [9]:
vectorizer = CountVectorizer(
    ngram_range=(1,2),   # учитываем пары слов
    min_df=10,            # пропускаем редкие токены
    max_df=0.85           # удаляем слишком частые
)

Усиление весов специфичных токенов

ClassTfidfTransformer(reduce_frequent_words=True) - дополнительно понижает вес избыточно частых слов внутри темы, что усиливает значимость специфичных терминов

In [10]:
ctfidf = ClassTfidfTransformer(reduce_frequent_words=True)  # дополнительная фильтрация часто встречающихся

In [11]:
# Сборный объект BERTopic
topic_model = BERTopic(
    embedding_model=embedder,
    umap_model=umap_model,
    hdbscan_model=clusterer,
    vectorizer_model=vectorizer,
    ctfidf_model=ctfidf,
    language='russian',
    calculate_probabilities=True,
    verbose=True
)

In [12]:
# Обучаем модель и получаем топики и вероятности
topics, probs = topic_model.fit_transform(cleaned_docs)

2025-05-01 18:36:29,732 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/3125 [00:00<?, ?it/s]

2025-05-01 18:44:04,047 - BERTopic - Embedding - Completed ✓
2025-05-01 18:44:04,048 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-05-01 18:45:32,460 - BERTopic - Dimensionality - Completed ✓
2025-05-01 18:45:32,461 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-05-01 18:46:03,046 - BERTopic - Cluster - Completed ✓
2025-05-01 18:46:03,055 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-05-01 18:46:21,910 - BERTopic - Representation - Completed ✓


# Оценка результатов

In [13]:
# Функция для подсчетв Topic Diversity
def calc_topic_diversity(model, top_n=10):
    all_terms = []
    for t_id, terms in model.get_topics().items():
        if t_id == -1:  # пропускаем шум
            continue
        all_terms += [term for term, _ in terms[:top_n]]
    return len(set(all_terms)) / len(all_terms)

In [14]:
diversity = calc_topic_diversity(topic_model)
print(f"topic_diversity: {diversity:.3f}")

topic_diversity: 0.886


In [15]:
# Рассчитаем когерентность UMass через Gensim
tokenized = [doc.split() for doc in cleaned_docs]
dict_gensim = Dictionary(tokenized)
corpus_gensim = [dict_gensim.doc2bow(doc) for doc in tokenized]

def calc_umass(model, texts, dictionary, corpus, top_n=10):
    keys = [ [w for w, _ in model.get_topic(t)[:top_n]]
             for t in model.get_topic_info().Topic if t != -1 ]
    cm = CoherenceModel(
        topics=keys,
        texts=texts,
        dictionary=dictionary,
        corpus=corpus,
        coherence='u_mass'
    )
    return cm.get_coherence()

In [16]:
umass = calc_umass(topic_model, tokenized, dict_gensim, corpus_gensim)
print(f"UMass Coherence: {umass:.3f}")

UMass Coherence: -2.313


In [24]:
# топ-токены для каждой темы
topic_info = topic_model.get_topic_info()
topic_info

,Topic,Count,Name,Representation,Representative_Docs
0,-1,23097,-1_суд_миллиард_закон_путин,"[суд, миллиард, закон, путин, федеральный, бан...",[бывший полицейский посадить смерть избитый ро...
1,0,49089,0_погибнуть_взрыв_полиция_задержать,"[погибнуть, взрыв, полиция, задержать, район, ...",[георгиевский лента грузинка спровоцировать де...
2,1,2668,1_матч_сборная_клуб_команда,"[матч, сборная, клуб, команда, футболист, чемп...",[капитан зенит пропустить старт лига чемпион к...
3,2,1806,2_аль_сирия_сирийский_ирак,"[аль, сирия, сирийский, ирак, иго, аль каеда, ...",[рука бин ладен саад хусейн хотеть совершить п...
4,3,1747,3_матч_сборная_команда_тренер,"[матч, сборная, команда, тренер, клуб, чемпион...",[испанский барселона выиграть лига чемпион пар...
...,...,...,...,...,...
68,67,70,67_промах_гонка_кубок мир_этап кубок,"[промах, гонка, кубок мир, этап кубок, секунда...",[ольга зайцев победить спринт этап кубок мир р...
69,68,67,68_мвф_транш_долг_миллиард доллар,"[мвф, транш, долг, миллиард доллар, кредит, ук...",[украина потратить деньга мвф погашение старый...
70,69,64,69_ипотечный_ипотека_кредит_ипотечный кредит,"[ипотечный, ипотека, кредит, ипотечный кредит,...",[россиянин пообещать триллион рубль жильё год ...
71,70,62,70_церковь_патриарх_православный церковь_право...,"[церковь, патриарх, православный церковь, прав...",[собор рекомендовать лишить сан епископ диомид...


In [18]:
# подробные токены по теме 0
print("Топ-токены темы 0:", topic_model.get_topic(0))

Топ-токены темы 0: [('погибнуть', 0.10804049506118837), ('взрыв', 0.10330923345889814), ('полиция', 0.10203058669404487), ('задержать', 0.10078543403291444), ('район', 0.10025930906838579), ('самолёт', 0.100108738252826), ('военный', 0.09833035077476776), ('мужчина', 0.09762549316735851), ('пожар', 0.09760702429235552), ('здание', 0.09719082128298022)]


In [ ]:
# визуализация документов в 2D пространстве
fig_docs = topic_model.visualize_documents(cleaned_docs)
fig_docs.show()

In [ ]:
# Распределение вероятностей тем для примера документа, берем документ с индексом 17
doc_id = 19
print("Документ:", cleaned_docs[doc_id][:200], "...")
fig_dist = topic_model.visualize_distribution(probs[doc_id], min_probability=0.02)
fig_dist.show()

In [23]:
#количество тем
print("Количество тем:", len(topic_model.get_topic_info().Topic.unique()) - 1)  # -1, т.к. шум

Количество тем: 72


In [26]:
noise_count = int(np.sum(np.array(topics) == -1))
total_docs = len(topics)
noise_ratio = noise_count / total_docs
print(f"Количество документов в кластерe -1 (шумные): {noise_count} из {total_docs} ({noise_ratio:.2%})")

Количество документов в кластерe -1 (шумные): 23097 из 100000 (23.10%)


# Выводы

- **Когерентность (UMass ≈ –2.31)**  - темы в целом логичны, но остаются размытые группы, где ключевые слова встречаются в разных контекстах

- **Разнообразие (≈ 0.89)** - большинство терминов уникальны, но 11 % пересечений указывает на смежные области

- **Шум и выбросы** - отдельные документы попадают в кластер «–1» (шум) из-за редкой лексики или отсутствия фокусной тематики

- **что можно попробовать для улучшения**  
  - Добавить POS-фильтрацию для точных биграмм/триграмм.  
  - Корректировать HDBSCAN (min_cluster_size, метрику) и UMAP (n_components).  
  - Проверить более глубокие RuBERT-модели для эмбеддингов.  